# ETL Process: Data Warehouse Implementation

This notebook implements the complete ETL process to transform cleaned source data into a star schema data warehouse.

## Process Overview
1. **Extract**: Load cleaned source data
2. **Transform**: Create dimensional and fact tables
3. **Validate**: Quality checks and referential integrity
4. **Load**: Export to CSV and generate reports

## Setup: Imports and Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_execution.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
SOURCE_DIR = "."
OUTPUT_DIR = "./dw_output"
DIMS_DIR = os.path.join(OUTPUT_DIR, "dimensions")
FACTS_DIR = os.path.join(OUTPUT_DIR, "facts")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

# Create output directories
for dir_path in [DIMS_DIR, FACTS_DIR, LOGS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

logger.info("ETL Process Started")
logger.info(f"Output directories created: {OUTPUT_DIR}")

2026-04-03 22:20:38,808 - INFO - ETL Process Started
2026-04-03 22:20:38,809 - INFO - Output directories created: ./dw_output


## Phase 1: Extraction

Load all cleaned source datasets

In [6]:
# Load source data
logger.info("=" * 60)
logger.info("PHASE 1: EXTRACTION")
logger.info("=" * 60)

try:
    df_pop = pd.read_csv("cleaned_ine_population_data.csv")
    logger.info(f"✓ Population data loaded: {len(df_pop)} rows")
    
    df_substations = pd.read_csv("cleaned_postos-transformacao-distribuicao.csv")
    logger.info(f"✓ Substations data loaded: {len(df_substations)} rows")
    
    df_weather_sp = pd.read_csv("cleaned_porto_serra_do_pilar_weather.csv")
    logger.info(f"✓ Weather (Serra do Pilar) loaded: {len(df_weather_sp)} rows")
    
    df_weather_pr = pd.read_csv("cleaned_porto_pedras_rubras_weather.csv")
    logger.info(f"✓ Weather (Pedras Rubras) loaded: {len(df_weather_pr)} rows")
    
    df_consumption = pd.read_csv("cleaned_consumo_horario_mobilidade_eletrica.csv")
    logger.info(f"✓ Consumption data loaded: {len(df_consumption)} rows")
    
    df_charging = pd.read_csv("cleaned_ine_charging_points_number_location.csv")
    logger.info(f"✓ Charging data loaded: {len(df_charging)} rows")
    
    logger.info("Extraction completed successfully")
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
    raise

2026-04-03 22:31:30,877 - INFO - ============================================================
2026-04-03 22:31:30,880 - INFO - PHASE 1: EXTRACTION
2026-04-03 22:31:30,882 - INFO - ============================================================
2026-04-03 22:31:30,894 - INFO - ✓ Population data loaded: 73 rows
2026-04-03 22:31:30,979 - INFO - ✓ Substations data loaded: 8412 rows
2026-04-03 22:31:30,997 - INFO - ✓ Weather (Serra do Pilar) loaded: 1520 rows
2026-04-03 22:31:31,015 - INFO - ✓ Weather (Pedras Rubras) loaded: 1520 rows
2026-04-03 22:31:33,491 - INFO - ✓ Consumption data loaded: 1027232 rows
2026-04-03 22:31:33,505 - INFO - ✓ Charging data loaded: 4320 rows
2026-04-03 22:31:33,506 - INFO - Extraction completed successfully


## Phase 2: Transformation

### 2.1 Create DIM_LOCATION

In [3]:
logger.info("=" * 60)
logger.info("PHASE 2: TRANSFORMATION")
logger.info("=" * 60)

# Extract unique municipalities from population data
dim_location = df_pop[['region_code', 'region_name']].drop_duplicates().reset_index(drop=True)

# Rename columns to match dimension schema
dim_location.columns = ['municipality_code', 'municipality_name']

# Assign location_id (surrogate key)
dim_location['location_id'] = range(1, len(dim_location) + 1)

# Add district information (derived from municipality name or mapping)
# For simplicity, we'll use municipality as both municipality and district
# In real scenario, you'd use a mapping table
dim_location['district_code'] = dim_location['municipality_code']
dim_location['district_name'] = dim_location['municipality_name']
dim_location['parish_code'] = None  # Optional data not available in cleaned sources
dim_location['parish_name'] = None

# Reorder columns according to schema
dim_location = dim_location[[
    'location_id',
    'municipality_code',
    'municipality_name',
    'parish_code',
    'parish_name',
    'district_code',
    'district_name'
]]

logger.info(f"✓ DIM_LOCATION created: {len(dim_location)} rows")
display(dim_location.head(10))

2026-04-03 22:21:54,948 - INFO - ============================================================
2026-04-03 22:21:54,952 - INFO - PHASE 2: TRANSFORMATION
2026-04-03 22:21:54,954 - INFO - ============================================================
2026-04-03 22:21:54,989 - INFO - ✓ DIM_LOCATION created: 19 rows


,location_id,municipality_code,municipality_name,parish_code,parish_name,district_code,district_name
0,1,NaN,NaN,None,None,NaN,NaN
1,2,11A,Área Metropolitana do Porto,None,None,11A,Área Metropolitana do Porto
2,3,11A0104,Arouca,None,None,11A0104,Arouca
3,4,11A0107,Espinho,None,None,11A0107,Espinho
4,5,11A1304,Gondomar,None,None,11A1304,Gondomar
5,6,11A1306,Maia,None,None,11A1306,Maia
6,7,11A1308,Matosinhos,None,None,11A1308,Matosinhos
7,8,11A0113,Oliveira de Azeméis,None,None,11A0113,Oliveira de Azeméis
8,9,11A1310,Paredes,None,None,11A1310,Paredes
9,10,11A1312,Porto,None,None,11A1312,Porto


### 2.2 Create DIM_TIME

In [8]:
# Extract min and max dates from all datasets
dates = []

# From population: year
pop_dates = pd.to_datetime(df_pop['year'].astype(str))
dates.extend(pop_dates.tolist())

# From weather
if 'date' in df_weather_sp.columns:
    dates.extend(pd.to_datetime(df_weather_sp['date']).tolist())
if 'date' in df_weather_pr.columns:
    dates.extend(pd.to_datetime(df_weather_pr['date']).tolist())

# From consumption
if 'Period' in df_consumption.columns:
    dates.extend(pd.to_datetime(df_consumption['Period'], errors='coerce').dropna().tolist())

# From charging
charging_dates = pd.to_datetime(
    df_charging['year'].astype(str) + '-' + 
    df_charging['month'].astype(str) + '-01',
    errors='coerce'
)
dates.extend(charging_dates.tolist())

# Remove None and duplicates
dates = pd.to_datetime([d for d in dates if d is not None]).unique()
dates = sorted(dates)

min_date = pd.Timestamp(dates[0])
max_date = pd.Timestamp(dates[-1])

logger.info(f"Date range: {min_date.date()} to {max_date.date()}")

# Generate complete datetime range
date_range = pd.date_range(start=min_date, end=max_date, freq='H')

dim_time = pd.DataFrame({
    'datetime': date_range
})

# Extract datetime components
dim_time['date'] = dim_time['datetime'].dt.date
dim_time['year'] = dim_time['datetime'].dt.year
dim_time['month'] = dim_time['datetime'].dt.month
dim_time['day'] = dim_time['datetime'].dt.day
dim_time['hour'] = dim_time['datetime'].dt.hour

# Create level keys (surrogate keys for each level)
# Year level key
dim_time['time_id'] = dim_time['year'] - dim_time['year'].min() + 1

# Month level key
dim_time['month_id'] = (
    (dim_time['year'] - dim_time['year'].min()) * 12 + dim_time['month']
)

# Day level key
dim_time['day_id'] = (
    dim_time['datetime'].astype('int64') // (10**9 * 86400)
)

# Hour level key
dim_time['hour_id'] = range(len(dim_time))

# Reorder columns
dim_time = dim_time[[
    'hour_id',
    'day_id',
    'month_id',
    'time_id',
    'hour',
    'day',
    'month',
    'year',
    'date',
    'datetime'
]]

logger.info(f"✓ DIM_TIME created: {len(dim_time)} rows")
logger.info(f"  Time range: {dim_time['datetime'].min()} to {dim_time['datetime'].max()}")
display(dim_time.head(10))

ValueError: time data "Taxa de crescimento efetivo (%) por Local de residência (NUTS - 2024) -  Anual" doesn't match format "%Y", at position 4. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

### 2.3 Create DIM_WEATHER

In [ ]:
# Combine weather data from both stations
df_weather_sp['date'] = pd.to_datetime(df_weather_sp['date'])
df_weather_pr['date'] = pd.to_datetime(df_weather_pr['date'])

# Merge both weather sources (average their values)
weather_merged = pd.merge(
    df_weather_sp[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    df_weather_pr[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    on='date',
    how='outer',
    suffixes=('_sp', '_pr')
)

# Calculate averages
dim_weather = pd.DataFrame({
    'date': weather_merged['date'],
    'avg_temperature': weather_merged[['tavg_sp', 'tavg_pr']].mean(axis=1),
    'min_temperature': weather_merged[['tmin_sp', 'tmin_pr']].min(axis=1),
    'max_temperature': weather_merged[['tmax_sp', 'tmax_pr']].max(axis=1),
    'precipitation': weather_merged[['prcp_sp', 'prcp_pr']].sum(axis=1),
    'wind_speed': weather_merged[['wspd_sp', 'wspd_pr']].mean(axis=1),
    'pressure': weather_merged[['pres_sp', 'pres_pr']].mean(axis=1)
})

# Link to time dimension (day level)
dim_weather = dim_weather.merge(
    dim_time[['date', 'day_id', 'month_id', 'time_id']].drop_duplicates(),
    on='date',
    how='left'
)

# Assign weather_id
dim_weather['weather_id'] = range(1, len(dim_weather) + 1)
dim_weather.rename(columns={'time_id': 'time_id_ref'}, inplace=True)
dim_weather['time_id'] = dim_weather['day_id']  # Reference day level key

# Reorder columns
dim_weather = dim_weather[[
    'weather_id',
    'time_id',
    'avg_temperature',
    'min_temperature',
    'max_temperature',
    'precipitation',
    'wind_speed',
    'pressure'
]]

logger.info(f"✓ DIM_WEATHER created: {len(dim_weather)} rows")
display(dim_weather.head(10))

### 2.4 Create DIM_SUBSTATIONS

In [ ]:
# Process substations data
dim_substations = df_substations.copy()

# Assign substation_id (surrogate key)
dim_substations['substation_id'] = range(1, len(dim_substations) + 1)

# Rename and map columns to schema
dim_substations = dim_substations.rename(columns={
    'Code': 'installation_code',
    'Municipality': 'municipality_name'
})

# Map to location dimension to get municipality_code
dim_substations = dim_substations.merge(
    dim_location[['municipality_code', 'municipality_name', 'district_name']],
    on='municipality_name',
    how='left'
)

# Assign level keys (using month_id as proxy for data level)
dim_substations['municipality_code_lk'] = dim_substations['municipality_code']

# Select and reorder columns
columns_to_keep = [
    'substation_id',
    'installation_code',
    'municipality_code_lk',
    'municipality_name',
    'district_name'
]

# Add optional columns if they exist
optional_cols = [
    'Latitude', 'Longitude',  # Geographic coordinates
    'Installed Power (kVA)',
    'Contract power (kVA)',
    'Usage level (%)',
    'Number of Clients',
    'Construction type'
]

for col in optional_cols:
    if col in dim_substations.columns:
        columns_to_keep.append(col)

dim_substations = dim_substations[columns_to_keep]

# Rename to match schema
rename_map = {
    'municipality_code_lk': 'municipality_code',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Installed Power (kVA)': 'installed_power',
    'Contract power (kVA)': 'contracted_power',
    'Usage level (%)': 'usage_level',
    'Number of Clients': 'number_of_clients',
    'Construction type': 'construction_type'
}

dim_substations = dim_substations.rename(columns=rename_map)

logger.info(f"✓ DIM_SUBSTATIONS created: {len(dim_substations)} rows")
display(dim_substations.head(10))

### 2.5 Create FACT_CONSUMPTION

In [ ]:
# Process consumption data
fact_consumption = df_consumption.copy()

# Rename Period to datetime if exists
if 'Period' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Period'])
    fact_consumption = fact_consumption.drop('Period', axis=1)
elif 'Date' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Date'])
    fact_consumption = fact_consumption.drop('Date', axis=1)

# Extract hour information
fact_consumption['date'] = fact_consumption['datetime'].dt.date
fact_consumption['hour'] = fact_consumption['datetime'].dt.hour

# Map municipality to location_id
fact_consumption = fact_consumption.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='Municipality',
    right_on='municipality_name',
    how='left'
)

# Map datetime to time_id (hour level)
fact_consumption = fact_consumption.merge(
    dim_time[['datetime', 'hour_id', 'day_id']],
    on='datetime',
    how='left'
)

# Map to weather (using day_id)
fact_consumption = fact_consumption.merge(
    dim_weather[['day_id', 'weather_id']],
    on='day_id',
    how='left'
)

# For substations: left join (optional) - won't always match
fact_consumption = fact_consumption.merge(
    dim_substations[['municipality_name', 'substation_id']],
    on='municipality_name',
    how='left'
)

# Rename value column to energy_consumed
if 'Value' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Value'], errors='coerce')
elif 'value' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['value'], errors='coerce')

# Select fact table columns
fact_consumption = fact_consumption[[
    'location_id',
    'hour_id',
    'day_id',
    'weather_id',
    'substation_id',
    'energy_consumed'
]]

# Remove rows with missing key references
fact_consumption = fact_consumption.dropna(subset=['location_id', 'hour_id', 'energy_consumed'])

logger.info(f"✓ FACT_CONSUMPTION created: {len(fact_consumption)} rows")
display(fact_consumption.head(10))

### 2.6 Create FACT_CHARGING

In [ ]:
# Process charging data
fact_charging = df_charging.copy()

# Rename Value to energy_consumed
if 'Value' in fact_charging.columns:
    fact_charging['energy_consumed'] = pd.to_numeric(fact_charging['Value'], errors='coerce')
elif 'value' in fact_charging.columns:
    fact_charging['energy_consumed'] = pd.to_numeric(fact_charging['value'], errors='coerce')

# Map region to location_id
fact_charging = fact_charging.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='region',
    right_on='municipality_name',
    how='left'
)

# Create datetime from year and month
fact_charging['datetime'] = pd.to_datetime(
    fact_charging['year'].astype(str) + '-' +
    fact_charging['month'].astype(str) + '-01'
)

# Map to month level time_id
fact_charging = fact_charging.merge(
    dim_time[['datetime', 'month_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# Outlet_type should already exist in source
if 'Outlet_Type' not in fact_charging.columns and 'outlet_type' not in fact_charging.columns:
    fact_charging['outlet_type'] = 'Unknown'

# Select fact table columns
fact_charging = fact_charging[[
    'location_id',
    'month_id',
    'outlet_type',
    'energy_consumed'
]]

# Remove rows with missing key references
fact_charging = fact_charging.dropna(subset=['location_id', 'month_id', 'energy_consumed'])

logger.info(f"✓ FACT_CHARGING created: {len(fact_charging)} rows")
display(fact_charging.head(10))

### 2.7 Create FACT_POPULATION

In [ ]:
# Process population data
fact_population = df_pop.copy()

# Remove rows with missing year or region
fact_population = fact_population.dropna(subset=['year', 'region_name'])

# Map region to location_id
fact_population = fact_population.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='region_name',
    right_on='municipality_name',
    how='left'
)

# Create datetime from year (01/01)
fact_population['datetime'] = pd.to_datetime(
    fact_population['year'].astype(str) + '-01-01'
)

# Map to year level time_id
fact_population = fact_population.merge(
    dim_time[['datetime', 'time_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# Convert measures to numeric
fact_population['density'] = pd.to_numeric(fact_population['density'], errors='coerce')
fact_population['growth_rate'] = pd.to_numeric(fact_population['growth_rate'], errors='coerce')

# Select fact table columns
fact_population = fact_population[[
    'location_id',
    'time_id',
    'density',
    'growth_rate'
]]

# Remove rows with missing key references
fact_population = fact_population.dropna(subset=['location_id', 'time_id'])

logger.info(f"✓ FACT_POPULATION created: {len(fact_population)} rows")
display(fact_population.head(10))

## Phase 3: Validation

Data quality checks and referential integrity

In [ ]:
logger.info("=" * 60)
logger.info("PHASE 3: VALIDATION")
logger.info("=" * 60)

validation_report = []

def check_nulls(df, table_name, critical_columns):
    """Check for null values in critical columns"""
    null_count = df[critical_columns].isnull().sum()
    if null_count.sum() > 0:
        logger.warning(f"{table_name} - NULL values found:")
        for col, count in null_count[null_count > 0].items():
            logger.warning(f"  {col}: {count} nulls")
        validation_report.append(f"⚠ {table_name}: {null_count.sum()} null values")
    else:
        logger.info(f"✓ {table_name}: No null values in critical columns")
        validation_report.append(f"✓ {table_name}: No null values")

# Dimension validation
logger.info("\n--- DIMENSION VALIDATION ---")
check_nulls(dim_location, "DIM_LOCATION", ['location_id', 'municipality_code'])
check_nulls(dim_time, "DIM_TIME", ['time_id', 'year'])
check_nulls(dim_weather, "DIM_WEATHER", ['weather_id'])
check_nulls(dim_substations, "DIM_SUBSTATIONS", ['substation_id'])

# Fact validation
logger.info("\n--- FACT TABLE VALIDATION ---")
check_nulls(fact_consumption, "FACT_CONSUMPTION", ['location_id', 'hour_id', 'energy_consumed'])
check_nulls(fact_charging, "FACT_CHARGING", ['location_id', 'month_id', 'energy_consumed'])
check_nulls(fact_population, "FACT_POPULATION", ['location_id', 'time_id'])

In [ ]:
# Referential Integrity Checks
logger.info("\n--- REFERENTIAL INTEGRITY ---")

def check_foreign_keys(fact_df, dim_df, fact_col, dim_col, table_names):
    """Check if all FK values exist in dimension"""
    missing = fact_df[~fact_df[fact_col].isin(dim_df[dim_col])]
    if len(missing) > 0:
        logger.warning(f"{table_names[0]} → {table_names[1]}: {len(missing)} orphan records")
        validation_report.append(f"⚠ {table_names[0]} → {table_names[1]}: {len(missing)} orphan FK")
    else:
        logger.info(f"✓ {table_names[0]} → {table_names[1]}: FK integrity OK")
        validation_report.append(f"✓ {table_names[0]} → {table_names[1]}: FK OK ({len(fact_df)} rows)")

# Check consumption FKs
check_foreign_keys(fact_consumption, dim_location, 'location_id', 'location_id', 
                   ['FACT_CONSUMPTION', 'DIM_LOCATION'])
check_foreign_keys(fact_consumption, dim_time, 'hour_id', 'hour_id', 
                   ['FACT_CONSUMPTION', 'DIM_TIME'])
check_foreign_keys(fact_consumption, dim_weather, 'weather_id', 'weather_id', 
                   ['FACT_CONSUMPTION', 'DIM_WEATHER'])

# Check charging FKs
check_foreign_keys(fact_charging, dim_location, 'location_id', 'location_id', 
                   ['FACT_CHARGING', 'DIM_LOCATION'])
check_foreign_keys(fact_charging, dim_time, 'month_id', 'month_id', 
                   ['FACT_CHARGING', 'DIM_TIME'])

# Check population FKs
check_foreign_keys(fact_population, dim_location, 'location_id', 'location_id', 
                   ['FACT_POPULATION', 'DIM_LOCATION'])
check_foreign_keys(fact_population, dim_time, 'time_id', 'time_id', 
                   ['FACT_POPULATION', 'DIM_TIME'])

In [ ]:
# Summary Statistics
logger.info("\n--- SUMMARY STATISTICS ---")

summary = f"""
DIM_LOCATION:        {len(dim_location):>7} rows | PKs: {len(dim_location['location_id'].unique())} unique
DIM_TIME:            {len(dim_time):>7} rows | PKs: {len(dim_time['time_id'].unique())} unique
DIM_WEATHER:         {len(dim_weather):>7} rows | PKs: {len(dim_weather['weather_id'].unique())} unique
DIM_SUBSTATIONS:     {len(dim_substations):>7} rows | PKs: {len(dim_substations['substation_id'].unique())} unique

FACT_CONSUMPTION:    {len(fact_consumption):>7} rows | Energy: {fact_consumption['energy_consumed'].sum():.2f} kWh
FACT_CHARGING:       {len(fact_charging):>7} rows | Energy: {fact_charging['energy_consumed'].sum():.2f} kWh
FACT_POPULATION:     {len(fact_population):>7} rows | Locations: {len(fact_population['location_id'].unique())}
"""

logger.info(summary)
validation_report.append(summary)

## Phase 4: Loading

Export to CSV files

In [ ]:
logger.info("=" * 60)
logger.info("PHASE 4: LOADING")
logger.info("=" * 60)

# Export Dimensions
logger.info("\nExporting dimension tables...")
dim_location.to_csv(os.path.join(DIMS_DIR, 'dim_location.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_location.csv")

dim_time.to_csv(os.path.join(DIMS_DIR, 'dim_time.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_time.csv")

dim_weather.to_csv(os.path.join(DIMS_DIR, 'dim_weather.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_weather.csv")

dim_substations.to_csv(os.path.join(DIMS_DIR, 'dim_substations.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_substations.csv")

# Export Facts
logger.info("\nExporting fact tables...")
fact_consumption.to_csv(os.path.join(FACTS_DIR, 'fact_consumption.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_consumption.csv")

fact_charging.to_csv(os.path.join(FACTS_DIR, 'fact_charging.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_charging.csv")

fact_population.to_csv(os.path.join(FACTS_DIR, 'fact_population.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_population.csv")

logger.info("\n✓ All tables exported successfully")

In [ ]:
# Generate Validation Report
report_path = os.path.join(LOGS_DIR, 'etl_validation_report.txt')

with open(report_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("DATA WAREHOUSE ETL VALIDATION REPORT\n")
    f.write("="*60 + "\n\n")
    
    f.write(f"Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("VALIDATION RESULTS:\n")
    f.write("-" * 60 + "\n")
    for item in validation_report:
        f.write(str(item) + "\n")
    
    f.write("\n" + "-" * 60 + "\n")
    f.write(f"ETL EXECUTION: SUCCESS\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n")

logger.info(f"✓ Validation report saved: {report_path}")
logger.info("="*60)
logger.info("ETL PROCESS COMPLETED SUCCESSFULLY")
logger.info("="*60)

## Summary

### Dimensional Bus Matrix Implementation

| Data Mart | Star | Dimensions Used |
|-----------|------|------------------|
| Energy | Consumption | Location, Time, Weather, Substations |
| Mobility | Charging | Location, Time |
| Demographics | Population | Location, Time |

### Output Files

**Dimensions:**
- `dimensions/dim_location.csv` - Geographic hierarchy
- `dimensions/dim_time.csv` - Multi-level temporal dimension
- `dimensions/dim_weather.csv` - Daily weather conditions
- `dimensions/dim_substations.csv` - Electrical infrastructure

**Facts:**
- `facts/fact_consumption.csv` - Hourly energy consumption
- `facts/fact_charging.csv` - Monthly EV charging activity
- `facts/fact_population.csv` - Yearly demographic indicators

### Next Steps

1. **Load to Database**: Create SQL schema and load CSVs to database
2. **Add Indexes**: Create indexes on PK, FK, and commonly filtered columns
3. **OLAP Cubes**: Build aggregated views for reporting
4. **Test Queries**: Run dimensional analysis queries
5. **Documentation**: Create data dictionary and business glossary